# Build a Codebase Q&A Tool — Colab / Kaggle / Binder companion

This notebook mirrors the local `uv` project from the course's
[Build a Codebase Q&A Tool](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/codebase-qa)
lesson, adapted to run in a hosted notebook with no local files: a tiny sample repo is
written out inline, then both query modes run over it — exact symbol lookup via `ast`,
and semantic search via local embeddings with `sentence-transformers`.

See the [lesson](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/codebase-qa) for the full walkthrough and the
[local example project](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/examples/codebase-qa) for the real, file-based version of this same code.

## Step 0: Install dependencies

In [ ]:
!pip install sentence-transformers numpy openai

## Step 1: Sample repo

A hosted notebook has no local files to read, so the tiny sample training pipeline from
[`sample_repo/`](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/examples/codebase-qa/sample_repo) is written out inline. It's deliberately small so both query modes have obvious answers: a `train_model` function, a `split_data` helper, and an `evaluate` function.

In [ ]:
from pathlib import Path

SAMPLE = {
    "main.py": """from utils import split_data


def train_model(features, labels, epochs: int = 5):
    train_x, test_x, train_y, test_y = split_data(features, labels)
    for epoch in range(epochs):
        loss = 0.9 ** epoch
        print(f"epoch {epoch + 1}/{epochs}  loss={loss:.4f}")
    return {"epochs": epochs, "train_x": train_x, "test_x": test_x}


def evaluate(model, features, labels) -> float:
    return 0.97
""",
    "utils.py": """def split_data(features, labels, ratio: float = 0.8):
    n = len(features)
    cut = int(n * ratio)
    return (features[:cut], features[cut:], labels[:cut], labels[cut:])


def shuffle(data, seed: int = 42):
    return data
""",
}

REPO = Path("sample_repo")
REPO.mkdir(exist_ok=True)
for name, source in SAMPLE.items():
    (REPO / name).write_text(source)
print("wrote", len(SAMPLE), "files to", REPO)

## Step 2: Extract exact symbols with `ast`

Same index as [`symbols.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/codebase-qa/symbols.py):
walk only top-level statements of each file's AST, recording every function/class/method/import
with its line number. This answers "where is X defined?" exactly — no LLM, no API key.

In [ ]:
import ast
import json


def extract_symbols(path) -> list[dict]:
    source = path.read_text(encoding="utf-8", errors="replace")
    tree = ast.parse(source, filename=str(path))
    symbols = []
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            symbols.append({"kind": "function", "name": node.name, "line": node.lineno, "file": str(path)})
        elif isinstance(node, ast.ClassDef):
            symbols.append({"kind": "class", "name": node.name, "line": node.lineno, "file": str(path)})
            for item in node.body:
                if isinstance(item, (ast.FunctionDef, ast.AsyncFunctionDef)):
                    symbols.append({"kind": "method", "name": f"{node.name}.{item.name}", "line": item.lineno, "file": str(path)})
        elif isinstance(node, ast.Import):
            for alias in node.names:
                symbols.append({"kind": "import", "name": alias.name.split(".")[0], "line": node.lineno, "file": str(path)})
        elif isinstance(node, ast.ImportFrom):
            if node.module:
                symbols.append({"kind": "import", "name": node.module.split(".")[0], "line": node.lineno, "file": str(path)})
    return symbols


symbols = []
for path in sorted(REPO.rglob("*.py")):
    symbols.extend(extract_symbols(path))
print(f"Indexed {len(symbols)} symbols")
for s in symbols:
    print(f"  {s['kind']:9s} {s['name']} @ {s['file']}:{s['line']}")

## Step 3: Chunk and embed the repo

Same code-aware chunker and embedding model as the local example. Code splits at
function/class boundaries, docs by paragraph; every chunk carries a `file:start-end` range.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

SKIP_DIRS = {".git", "__pycache__", "venv", ".venv", "build", "node_modules"}
TARGET_CHUNK_SIZE = 500


def load_chunks(repo_root: Path) -> list[dict]:
    chunks = []
    for path in sorted(repo_root.rglob("*")):
        if path.is_dir():
            continue
        if any(part in SKIP_DIRS for part in path.relative_to(repo_root).parts):
            continue
        if path.suffix.lower() not in {".py", ".md", ".mdx", ".txt"}:
            continue
        text = path.read_text(encoding="utf-8", errors="replace")
        source = str(path.relative_to(repo_root))
        lines = text.splitlines()
        boundaries = [0]
        for i, line in enumerate(lines):
            if line.startswith(("def ", "class ")) and not line.startswith(("    ", "\t")):
                if i > 0:
                    boundaries.append(i)
        boundaries.append(len(lines))
        for start, end in zip(boundaries, boundaries[1:]):
            body = "\n".join(lines[start:end]).strip()
            if body:
                chunks.append({"source": source, "start": start + 1, "end": end, "text": body})
    return chunks


chunks = load_chunks(REPO)
MODEL_NAME = "all-MiniLM-L6-v2"
print(f"Embedding {len(chunks)} chunks with {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode([c["text"] for c in chunks], normalize_embeddings=True)
print(f"Embedded {embeddings.shape[0]} chunks ({embeddings.shape[1]}-dim)")

## Step 4: Exact lookup — "where is X defined?"

The dispatcher's exact mode: find every symbol whose name matches and show `file:line`. No
LLM involved.

In [ ]:
def find_symbol(name: str) -> list[dict]:
    return [s for s in symbols if s["name"] == name or name in s["name"]]


for s in find_symbol("split_data"):
    print(f"  [{s['kind']}] {s['name']} @ {s['file']}:{s['line']}")

## Step 5: Semantic search — "how does X work?"

The dispatcher's fuzzy mode: embed the question, rank chunks by cosine similarity, and show
the top hits with their `file:line` citations. No LLM needed to *see* what's relevant.

In [ ]:
def retrieve(question: str, top_k: int = 3) -> list[dict]:
    question_vector = model.encode([question], normalize_embeddings=True)[0]
    similarities = embeddings @ question_vector
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return [
        {**chunks[i], "score": float(similarities[i])}
        for i in top_indices
    ]


for r in retrieve("how does the training loop work?"):
    print(f"{r['score']:.3f}  [{r['source']}:{r['start']}-{r['end']}]")
    print(f"      {r['text'][:90]}...")

## Step 6: Ground an LLM answer (optional — needs a free-tier key)

Same prompt as [`query.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/codebase-qa/query.py):
hand the model the retrieved chunks and demand `[path.py:start-end]` citations, so the
answer points at the exact lines to verify.

In [ ]:
import getpass
import os
from openai import OpenAI

os.environ["GITHUB_TOKEN"] = getpass.getpass("Paste your GitHub Models API key: ")

client = OpenAI(
    api_key=os.environ["GITHUB_TOKEN"],
    base_url="https://models.github.ai/inference",
)

question = "how does the training loop work?"
context_blocks = [
    f"[{c['source']}:{c['start']}-{c['end']}]\n{c['text']}" for c in retrieve(question)
]
prompt = f"""Answer the question using ONLY the context below. Each block is
tagged with the file and line range it came from. Support every factual claim
with a citation in the form [path.py:start-end]. If the context doesn't contain
the answer, say so -- do not make something up.

Context:
{chr(10).join(context_blocks)}

Question: {question}

Answer:"""

response = client.chat.completions.create(
    model="gpt-4o-mini",  # confirm this still has a free tier before running
    messages=[{"role": "user", "content": prompt}],
)
print(response.choices[0].message.content)